# Exploratory Data Analysis & Stateful Velocity Features - Project 3
> **Exploratory Prototype Note:** This notebook explores raw SAML-D transaction distributions and 4 initial prototype features. The shipped production pipeline (`src/features/velocity.py` & `src/models/anomaly_detector.py`) computes all 7 stateful graph/velocity features (`CANONICAL_AML_FEATURES`) in `StreamingStateManager` and scores events with a supervised `RandomForestClassifier(n_estimators=50, max_depth=10, class_weight='balanced')`.

In [ ]:
import shutil
from pathlib import Path
import kagglehub
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "SAML-D.csv"

if not DATA_PATH.exists():
    dl_dir = Path(
        kagglehub.dataset_download(
            "berkanoztas/synthetic-transaction-monitoring-dataset-aml"
        )
    )
    csv_files = list(dl_dir.rglob("*.csv"))
    if csv_files:
        DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(csv_files[0], DATA_PATH)

df_sample = pd.read_csv(DATA_PATH, nrows=100000)
print(f"Loaded SAML-D prefix sample: {df_sample.shape}")

In [ ]:
laundering_counts = df_sample["Is_laundering"].value_counts()
print("Sample Class Counts:\n", laundering_counts)
df_sample["is_structuring"] = (
    (df_sample["Amount"] >= 8000.0) & (df_sample["Amount"] < 10000.0)
).astype(int)
print("Structuring band events ($8k-$10k):", int(df_sample["is_structuring"].sum()))